# pykrx data-depth probe (KRX login required; no OPEN API key needed)

**Purpose:** determine **how many years back** Korean prices, market cap, and PBR each reach.
This drives the sample period — in particular, if PBR reaches before 2015, there is a workaround
to build B/M (the value factor) over a longer history without OpenDART (financial statements 2015+).

> **2025-12-27 change:** the KRX information data system moved to a membership-based 'KRX Data Marketplace', so **login is now required** (data queries themselves remain free).
> Market cap, PBR, and all-ticker fundamentals return an empty response without login, which then fails inside pykrx with a JSON parse error.
> Complete the **0) Setup** cell below first, then **restart the kernel** and run from the top.
>
> After viewing the results, record the **earliest date and ticker count** for each probe. Those fix the sample period.

## 0) KRX sign-up + .env setup (one-time)

**Sign up (free):**
1. https://data.krx.co.kr -> **Sign up** (top right)
2. Individual member -> identity verification or social login
3. Use the resulting ID/PW as `KRX_ID` / `KRX_PW`

> Note: pykrx uses this **regular member account**. It is different from the official OPEN API auth key at `openapi.krx.co.kr`; no auth-key application is required.

**`.env` file (project root):**
```
KRX_ID=your_krx_id
KRX_PW=your_krx_password
```

**Additional setup:**
- `pip install python-dotenv`
- Add `.env` to `.gitignore` (so credentials are not committed to git)
- **After setup, restart the kernel and run from the top** — pykrx logs in at import time, so once imported it will not re-login even if .env is re-read.

## Prerequisites

`pip install pykrx pandas python-dotenv`

In [1]:
# 1) Load KRX credentials "before" importing pykrx.
#    pykrx attempts KRX login at import time, so this ordering matters.
import os
from dotenv import load_dotenv

load_dotenv()  # read KRX_ID, KRX_PW from the project-root .env
assert os.getenv('KRX_ID') and os.getenv('KRX_PW'), (
    'KRX_ID / KRX_PW are not set. Create a .env in the project root, fill in the values, '
    'then restart the kernel and run from the top.'
)

# 2) With credentials in the environment, import -> login succeeds
from pykrx import stock
import pandas as pd

TODAY = pd.Timestamp.today().strftime('%Y%m%d')
SAMSUNG = '005930'   # Samsung Electronics (listed 1975 — a deep-history reference ticker)
print('Login account:', os.getenv('KRX_ID')[:3] + '***')
print('Today:', TODAY)

KRX 로그인 시도...
  로그인 ID: ***
KRX 로그인 완료.
  로그인 시간: 2026-06-30 23:38:41
  만료 시간: 2026-07-01 00:38:41
로그인 계정: ***
오늘: 20260630


## 1) Price (OHLCV) depth

Request Samsung Electronics daily close from 1995 -> see the date where data actually begins.

> Note: OHLCV comes from a Naver source, so it works without login, but it may be capped at ~3000 historical rows (~12 years).
> If the start date stops near 2014, that is a source limit; for longer price history, fetch separately after login via `get_market_cap` (KRX source, includes close) or FinanceDataReader.

In [2]:
px = stock.get_market_ohlcv_by_date('19950101', TODAY, SAMSUNG)
print('Price data:', px.index.min(), '~', px.index.max(), '/', len(px), 'rows')
px.head(2)

가격 데이터: 2014-04-07 00:00:00 ~ 2026-06-30 00:00:00 / 3000 행


,시가,고가,저가,종가,거래량,등락률
날짜,,,,,,
2014-04-07,27940,27940,27480,27940,215235,NaN
2014-04-08,27740,27980,27500,27880,212164,-0.214746


## 2) Market-cap depth (for the size factor) <- login required

In [3]:
cap = stock.get_market_cap_by_date('19950101', TODAY, SAMSUNG)
print('Market-cap data:', cap.index.min(), '~', cap.index.max(), '/', len(cap), 'rows')
cap.head(2)

시총 데이터: 1995-05-02 00:00:00 ~ 2026-06-30 00:00:00 / 7852 행


,시가총액,거래량,거래대금,상장주식수
날짜,,,,
1995-05-02,6497053077500,139560,16676735000,54368645
1995-05-03,6714527657500,382980,47649710000,54368645


## 3) Fundamentals (PBR/BPS) depth <- key to the B/M workaround (login required)

How far back KRX's directly-disclosed PBR is valid. If this reaches before 2015, the sample gets longer.

In [4]:
fun = stock.get_market_fundamental_by_date('19950101', TODAY, SAMSUNG)
print('Fundamental columns:', list(fun.columns))
print('Full range:', fun.index.min(), '~', fun.index.max())
if 'PBR' in fun.columns:
    valid = fun[fun['PBR'] > 0]
    print('PBR valid start:', valid.index.min() if len(valid) else 'none')
else:
    print('No PBR column — check login state or the function return shape')
fun.head(2)

펜더멘털 컬럼: ['BPS', 'PER', 'PBR', 'EPS', 'DIV', 'DPS']
전체 범위: 1995-05-02 00:00:00 ~ 2026-06-30 00:00:00
PBR 유효 시작: 2002-04-23 00:00:00


,BPS,PER,PBR,EPS,DIV,DPS
날짜,,,,,,
1995-05-02,0,0.0,0.0,0,0.0,0
1995-05-03,0,0.0,0.0,0,0.0,0


## 4) Historical universe (ticker count) — KOSPI / KOSDAQ

Whether the ticker list comes back correctly at several past dates (i.e. whether the universe can be constructed at that point in time).

In [5]:
for d in ['20001229', '20051230', '20101230', '20151230', '20201230']:
    try:
        k = len(stock.get_market_ticker_list(d, market='KOSPI'))
        q = len(stock.get_market_ticker_list(d, market='KOSDAQ'))
        print(f'{d}:  KOSPI {k},  KOSDAQ {q}')
    except Exception as e:
        print(f'{d}:  failed — {type(e).__name__}: {e}')

20001229:  KOSPI 902개,  KOSDAQ 615개
20051230:  KOSPI 858개,  KOSDAQ 931개
20101230:  KOSPI 927개,  KOSDAQ 1035개
20151230:  KOSPI 887개,  KOSDAQ 1154개
20201230:  KOSPI 917개,  KOSDAQ 1471개


## 5) Whole-market PBR on a given date (is a B/M sort feasible) <- login required

We must be able to pull the whole market's PBR at once for a single date in order to sort into 25 buckets.

In [6]:
for d in ['20151230', '20101230', '20051230']:
    try:
        f = stock.get_market_fundamental_by_ticker(d, market='KOSPI')
        nvalid = int((f['PBR'] > 0).sum()) if 'PBR' in f.columns else 0
        print(f'{d} KOSPI:  {len(f)} tickers, {nvalid} with valid PBR')
    except Exception as e:
        print(f'{d}:  failed — {type(e).__name__}: {e}')

20151230 KOSPI:  854종목 중 PBR 유효 729개
20101230 KOSPI:  872종목 중 PBR 유효 702개
20051230 KOSPI:  819종목 중 PBR 유효 0개
